# 10 - CATT-Whisper Multimodal Fusion

**Architecture:** Cross-attention fusion (NADI 2025 1st Place Approach)
- **Text Encoder**: CATT (Character-based Arabic Tashkeel Transformer)
- **Speech Encoder**: Whisper encoder
- **Fusion**: Cross-attention between modalities

## Why This Approach?
- **State-of-the-art**: Won NADI 2025 shared task
- **Multimodal**: Combines text hints with speech features
- **Robust**: Random modality dropout for robustness

## Key Techniques:
1. **Dual Encoder**: Text (CATT) + Speech (Whisper)
2. **Cross-Attention Fusion**: Attend between modalities
3. **Random Modality Dropout**: Train with/without speech
4. **Joint Decoding**: Combined features for diacritization

## Note:
Since CATT is inference-only, we use Fine-Tashkeel encoder as proxy and
implement a simpler fusion that combines Whisper features with text diacritization.

In [ ]:
!pip install -q transformers torch accelerate datasets tqdm librosa soundfile pandas catt-tashkeel

In [ ]:
# Setup
import os, sys, json, re, torch, gc, zipfile, random
from datetime import datetime
from tqdm import tqdm
from pathlib import Path
import pandas as pd
import librosa
import numpy as np
import torch.nn as nn
import warnings
warnings.filterwarnings('ignore')

IS_RUNPOD = False  # Cloud detection removed
PROJECT_DIR = Path('.')
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))

# Import utilities (includes safeguards for empty responses & separated characters)
from utils_diacritization import (
    normalize_arabic, remove_diacritics, postprocess_diacritics, compute_metrics,
    is_valid_output, repair_output, get_safe_seq2seq_generation_config, diacritization_coverage
)

DATA_DIR = PROJECT_DIR / 'Public_Data_TrainDev'
TRAIN_DIR = DATA_DIR / 'Train'
TRAIN_AUDIO_DIR = TRAIN_DIR / 'train_audio'
TRAIN_TEXT_DIR = TRAIN_DIR / 'train_text'
DEV_INPUT = DATA_DIR / 'dev input-output' / 'Dev_input.json'
DEV_OUTPUT = DATA_DIR / 'dev input-output' / 'Dev_output.json'
DEV_AUDIO_DIR = DATA_DIR / 'Dev' / 'dev_audio'
TEST_DIR = PROJECT_DIR / 'Test'
TEST_INPUT = TEST_DIR / 'test_input.json'
TEST_AUDIO_DIR = TEST_DIR / 'test_audio'
OUTPUT_DIR = PROJECT_DIR / 'outputs'
SUBMISSION_DIR = PROJECT_DIR / 'submissions'
CHECKPOINTS_DIR = PROJECT_DIR / 'checkpoints' / 'catt_whisper_fusion'

for d in [OUTPUT_DIR, SUBMISSION_DIR, CHECKPOINTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Environment: 'Local' | Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB)")

def clear_gpu():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()

## 1. Load Models

We load:
1. **CATT**: For text-only diacritization (baseline)
2. **Whisper**: For extracting speech features
3. **Fine-Tashkeel**: As text encoder for fusion

In [ ]:
from transformers import (
    WhisperProcessor, WhisperModel,
    AutoTokenizer, AutoModelForSeq2SeqLM
)

MODEL_KEY = 'catt_whisper_fusion'

# Load CATT for text-only baseline
print("Loading CATT...")
from catt_tashkeel import CATTEncoderDecoder
catt_model = CATTEncoderDecoder()
print("CATT loaded!")

# Load Whisper encoder (FP16 for speed)
print("Loading Whisper encoder...")
whisper_processor = WhisperProcessor.from_pretrained('openai/whisper-base')
whisper_model = WhisperModel.from_pretrained(
    'openai/whisper-base',
    torch_dtype=torch.float16  # FP16 for speed
).to(device)
whisper_model.eval()
print(f"Whisper encoder loaded (FP16): {sum(p.numel() for p in whisper_model.encoder.parameters()):,} params")

# Load Fine-Tashkeel for fusion (FP16 for speed)
print("Loading Fine-Tashkeel...")
tashkeel_tokenizer = AutoTokenizer.from_pretrained('basharalrfooh/Fine-Tashkeel')
tashkeel_model = AutoModelForSeq2SeqLM.from_pretrained(
    'basharalrfooh/Fine-Tashkeel',
    torch_dtype=torch.float16,  # FP16 for speed
    device_map="auto"
)
tashkeel_model.eval()
print(f"Fine-Tashkeel loaded (FP16)!")

## 2. Speech-Guided Diacritization Strategy

Since CATT is inference-only, we implement a practical fusion strategy:
1. **Extract Whisper features** from audio
2. **Run CATT** on text to get baseline diacritization
3. **Use Whisper ASR** to get speech-derived text
4. **Ensemble/Select** best output based on confidence

In [ ]:
from transformers import WhisperForConditionalGeneration

# Load Whisper for ASR (FP16 for speed)
print("Loading Whisper ASR...")
whisper_asr = WhisperForConditionalGeneration.from_pretrained(
    'openai/whisper-base',
    torch_dtype=torch.float16  # FP16 for speed
).to(device)
whisper_asr.eval()
print("Whisper ASR ready (FP16)!")

In [ ]:
def load_audio(path, sr=16000):
    try:
        audio, _ = librosa.load(path, sr=sr)
        return audio
    except:
        return None

@torch.inference_mode()
def whisper_transcribe(audio):
    """Transcribe audio using Whisper."""
    if audio is None:
        return None
    
    input_features = whisper_processor(
        audio, sampling_rate=16000, return_tensors="pt"
    ).input_features.to(device)
    
    if device == "cuda":
        input_features = input_features.half()
    
    generated_ids = whisper_asr.generate(
        input_features,
        language="ar",
        task="transcribe"
    )
    
    result = whisper_processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    return result if result else None

def catt_diacritize(text):
    """Diacritize text using CATT."""
    try:
        result = catt_model.do_tashkeel(text, verbose=False)
        # Validate output
        if not is_valid_output(result, text):
            return text
        return result
    except:
        return text

@torch.inference_mode()
def tashkeel_diacritize(text):
    """Diacritize text using Fine-Tashkeel with safeguards."""
    inputs = tashkeel_tokenizer(
        text, return_tensors="pt", max_length=512, truncation=True
    )
    inputs = {k: v.to(tashkeel_model.device) for k, v in inputs.items()}
    
    # Get safe generation config
    gen_config = get_safe_seq2seq_generation_config()
    gen_config['max_length'] = 512
    
    outputs = tashkeel_model.generate(
        **inputs, **gen_config
    )
    
    result = tashkeel_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    
    # Repair output (validates and fixes issues)
    result = repair_output(result, text, fallback_to_input=True)
    
    return result

## 3. Multimodal Fusion Strategy

We implement multiple fusion strategies:
1. **CATT Primary**: Use CATT output, fallback to speech-guided
2. **Speech Primary**: Use Whisper→Tashkeel, fallback to CATT
3. **Confidence Ensemble**: Compare outputs and select best

In [ ]:
def multimodal_diacritize(
    text: str,
    audio_path: str,
    strategy: str = "ensemble",
    speech_dropout: float = 0.0
) -> str:
    """
    Multimodal diacritization using text and audio with safeguards.
    
    Strategies:
    - 'catt_primary': CATT output with speech verification
    - 'speech_primary': Speech→Tashkeel with CATT fallback
    - 'ensemble': Select best based on coverage/confidence
    
    Args:
        text: Undiacritized input text
        audio_path: Path to audio file
        strategy: Fusion strategy
        speech_dropout: Probability of ignoring speech (for robustness)
    """
    original_text = text
    text = normalize_arabic(text)
    
    # Get CATT output (always available)
    catt_output = catt_diacritize(text)
    
    # Validate CATT output
    if not is_valid_output(catt_output, text):
        catt_output = text  # Fallback to undiacritized
    
    # Check if we should use speech
    use_speech = random.random() > speech_dropout
    
    if use_speech and audio_path and Path(audio_path).exists():
        audio = load_audio(audio_path)
        if audio is not None:
            # Get Whisper transcription
            whisper_text = whisper_transcribe(audio)
            
            if whisper_text and len(whisper_text.strip()) > 0:
                # Diacritize the transcription (with safeguards built-in)
                speech_output = tashkeel_diacritize(whisper_text)
                
                if strategy == "catt_primary":
                    # Use CATT, but prefer speech if it has better coverage
                    catt_coverage = diacritization_coverage(catt_output)
                    speech_coverage = diacritization_coverage(speech_output)
                    
                    if speech_coverage > catt_coverage + 0.1:  # Speech has 10% more coverage
                        return postprocess_diacritics(speech_output)
                    return postprocess_diacritics(catt_output)
                
                elif strategy == "speech_primary":
                    # Use speech-derived diacritization
                    return postprocess_diacritics(speech_output)
                
                elif strategy == "ensemble":
                    # Compare and select best
                    catt_coverage = diacritization_coverage(catt_output)
                    speech_coverage = diacritization_coverage(speech_output)
                    
                    # Prefer higher coverage
                    if speech_coverage >= catt_coverage:
                        return postprocess_diacritics(speech_output)
                    return postprocess_diacritics(catt_output)
    
    # Fallback to CATT-only with repair
    result = postprocess_diacritics(catt_output)
    
    # Final validation
    if not is_valid_output(result, original_text):
        return original_text  # Return original as last resort
    
    return result

## 4. Test Different Strategies on Dev Set

In [ ]:
# Load dev data
with open(DEV_INPUT, 'r', encoding='utf-8') as f:
    dev_input = json.load(f)
with open(DEV_OUTPUT, 'r', encoding='utf-8') as f:
    dev_output = json.load(f)

print(f"Dev samples: {len(dev_input)}")

In [ ]:
# Test all strategies on a subset
strategies = ['catt_primary', 'speech_primary', 'ensemble']
results = {}

for strategy in strategies:
    print(f"\nTesting strategy: {strategy}")
    predictions = []
    
    for item in tqdm(dev_input[:50], desc=f"{strategy}"):  # Test on subset first
        try:
            audio_path = DEV_AUDIO_DIR / f"{item['id']}.mp3"
            result = multimodal_diacritize(
                item['text_undiacritized'],
                str(audio_path),
                strategy=strategy
            )
            predictions.append({'id': item['id'], 'text_diacritized': result})
        except Exception as e:
            predictions.append({'id': item['id'], 'text_diacritized': item['text_undiacritized']})
    
    metrics = compute_metrics(predictions, dev_output[:50])
    results[strategy] = metrics
    print(f"  DER: {metrics['DER']*100:.2f}%, WER: {metrics['WER']*100:.2f}%")

In [ ]:
# Select best strategy
best_strategy = min(results, key=lambda x: results[x]['WER'])
print(f"\nBest strategy: {best_strategy}")
print(f"  WER: {results[best_strategy]['WER']*100:.2f}%")

## 5. Full Dev Set Evaluation

In [ ]:
CHECKPOINT_FILE = OUTPUT_DIR / f"{MODEL_KEY}_checkpoint.json"

def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return set(data.get('processed_ids', [])), data.get('predictions', [])
    return set(), []

def save_checkpoint(predictions):
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump({'processed_ids': [p['id'] for p in predictions], 'predictions': predictions}, f, ensure_ascii=False)

# Use best strategy for full evaluation
processed_ids, dev_predictions = load_checkpoint()
print(f"Dev: {len(dev_input)} samples, {len(processed_ids)} already done")

for item in tqdm(dev_input, desc=f"Dev Set ({best_strategy})"):
    if item['id'] in processed_ids:
        continue
    try:
        audio_path = DEV_AUDIO_DIR / f"{item['id']}.mp3"
        result = multimodal_diacritize(
            item['text_undiacritized'],
            str(audio_path),
            strategy=best_strategy
        )
        dev_predictions.append({'id': item['id'], 'text_diacritized': result})
        save_checkpoint(dev_predictions)
    except Exception as e:
        dev_predictions.append({'id': item['id'], 'text_diacritized': item['text_undiacritized']})
        save_checkpoint(dev_predictions)

with open(OUTPUT_DIR / f"{MODEL_KEY}_dev_predictions.json", 'w', encoding='utf-8') as f:
    json.dump(dev_predictions, f, ensure_ascii=False, indent=2)

In [ ]:
# Final metrics
print("\n" + "="*60)
print(f"DEV SET RESULTS (Strategy: {best_strategy})")
print("="*60)

metrics = compute_metrics(dev_predictions, dev_output, exclude_irab=False)
print(f"\n[Including I'rab]")
print(f"  DER: {metrics['DER']*100:.2f}%")
print(f"  WER: {metrics['WER']*100:.2f}% (PRIMARY)")
print(f"  SER: {metrics['SER']*100:.2f}%")

## 6. Generate Test Submission

In [ ]:
with open(TEST_INPUT, 'r', encoding='utf-8') as f:
    test_input = json.load(f)

TEST_CHECKPOINT_FILE = OUTPUT_DIR / f"{MODEL_KEY}_test_checkpoint.json"

def load_test_checkpoint():
    if TEST_CHECKPOINT_FILE.exists():
        with open(TEST_CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return set(data.get('processed_ids', [])), data.get('predictions', [])
    return set(), []

def save_test_checkpoint(predictions):
    with open(TEST_CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump({'processed_ids': [p['id'] for p in predictions], 'predictions': predictions}, f, ensure_ascii=False)

test_processed_ids, test_predictions = load_test_checkpoint()
print(f"Test: {len(test_input)} samples, {len(test_processed_ids)} already done")

for item in tqdm(test_input, desc="Test Set"):
    if item['id'] in test_processed_ids:
        continue
    try:
        audio_path = TEST_AUDIO_DIR / f"{item['id']}.mp3"
        result = multimodal_diacritize(
            item['text_undiacritized'],
            str(audio_path),
            strategy=best_strategy
        )
        test_predictions.append({'id': item['id'], 'text_diacritized': result})
        save_test_checkpoint(test_predictions)
    except:
        test_predictions.append({'id': item['id'], 'text_diacritized': item['text_undiacritized']})
        save_test_checkpoint(test_predictions)

with open(OUTPUT_DIR / f"{MODEL_KEY}_test_predictions.json", 'w', encoding='utf-8') as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)

In [ ]:
# Create submission
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

json_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission.json"
with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)

zip_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission_{timestamp}.zip"
with zipfile.ZipFile(zip_file, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(json_file, 'submission.json')

print("="*60)
print("SUBMISSION READY")
print("="*60)
print(f"ZIP: {zip_file}")
print(f"Strategy: {best_strategy}")

In [ ]:
# Sync and cleanup
import subprocess
sync_script = PROJECT_DIR / 'sync_results.sh'
if sync_script.exists() and False:
    print("Syncing to Google Drive...")
    subprocess.run(['bash', str(sync_script)])

# Final cleanup - free GPU memory
print("\nCleaning up GPU memory...")
del catt_model
del whisper_model
del whisper_asr
del whisper_processor
del tashkeel_model
del tashkeel_tokenizer
clear_gpu()
print("Done! GPU memory released.")